In [1]:
import json
from pathlib import Path

from app.intent import detect_intent
from app.tfidf_search import search

TEST_QUESTIONS_FILE = Path("data/test_questions.json")

with TEST_QUESTIONS_FILE.open("r", encoding="utf-8") as file:
    test_questions = json.load(file)

print(f"Liczba pytań testowych: {len(test_questions)}")


Liczba pytań testowych: 30


## Ewaluacja Intencji

In [2]:
correct = 0

for item in test_questions:
    question = item["question"]
    expected = item["expected_intent"]

    predicted = detect_intent(question)

    if predicted == expected:
        correct += 1
        status = "OK"
    else:
        status = "BŁĄD"

    print(
        f"{status:5} | "
        f"expected={expected:12} | "
        f"predicted={predicted:12} | "
        f"{question}"
    )

accuracy = correct / len(test_questions)

print()
print(f"Accuracy intencji = {accuracy:.3f}")
print(f"Accuracy intencji = {accuracy * 100:.1f}%")

OK    | expected=rekrutacja   | predicted=rekrutacja   | Jak wygląda rekrutacja na studia pierwszego stopnia?
OK    | expected=rekrutacja   | predicted=rekrutacja   | Jakie dokumenty muszę złożyć podczas rekrutacji?
OK    | expected=rekrutacja   | predicted=rekrutacja   | Kiedy rozpoczyna się rekrutacja na studia?
OK    | expected=rekrutacja   | predicted=rekrutacja   | Jakie są progi punktowe na informatykę?
OK    | expected=rekrutacja   | predicted=rekrutacja   | Jak obliczane są punkty podczas rekrutacji?
OK    | expected=rekrutacja   | predicted=rekrutacja   | Ile wynosi opłata rekrutacyjna?
OK    | expected=rekrutacja   | predicted=rekrutacja   | Ile miejsc jest dostępnych na kierunkach?
OK    | expected=studia       | predicted=studia       | Jakie kierunki studiów oferuje Politechnika Białostocka?
OK    | expected=studia       | predicted=studia       | Gdzie znajdę informacje o studiach pierwszego stopnia?
OK    | expected=studia       | predicted=studia       | Czy Politechnik

## Ewaluacja Wyszukiwania (Recall@5)

In [3]:
K = 5
correct = 0

for item in test_questions:
    question = item["question"]
    expected_urls = item["expected_url_contains"]
    if isinstance(expected_urls, str):
        expected_urls = [expected_urls]
    expected_urls = [url.lower() for url in expected_urls]

    results = search(question, k=K)

    found = any(
        expected_url in result["url"].lower()
        for expected_url in expected_urls
        for result in results
    )

    if found:
        correct += 1
        status = "OK"
    else:
        status = "BŁĄD"

    print(f"{status:5} | {question}")
    print(f"expected url contains one of: {expected_urls}")
    print("returned urls:")

    for result in results:
        print(f" - {result['url']} | score={result['score']:.3f}")

    print()

recall_at_5 = correct / len(test_questions)

print(f"Recall@{K} = {recall_at_5:.3f}")
print(f"Recall@{K} = {recall_at_5 * 100:.1f}%")

OK    | Jak wygląda rekrutacja na studia pierwszego stopnia?
expected url contains one of: ['rekrutacja-krok-po-kroku-studia-i-stopnia']
returned urls:
 - https://kandydacipb.edu.pl/studia-i-stopnia/rekrutacja-krok-po-kroku-studia-i-stopnia | score=0.364
 - https://kandydacipb.edu.pl/rekrutacja/studia-ii-stopnia/rekrutacja-krok-po-kroku-studia-ii-stopnia | score=0.315
 - https://kandydacipb.edu.pl/rekrutacja/studia-ii-stopnia | score=0.205
 - https://kandydacipb.edu.pl/rekrutacja/potwierdzenie-efektow-uczenia-sie/wykaz-kierunkow | score=0.193
 - https://kandydacipb.edu.pl/rekrutacja/potwierdzenie-efektow-uczenia-sie/wykaz-kierunkow-2 | score=0.193

OK    | Jakie dokumenty muszę złożyć podczas rekrutacji?
expected url contains one of: ['dokumenty-rekrutacyjne']
returned urls:
 - https://kandydacipb.edu.pl/studia-i-stopnia/dokumenty-rekrutacyjne | score=0.229
 - https://kandydacipb.edu.pl/rekrutacja/studia-ii-stopnia/dokumenty-rekrutacyjne | score=0.220
 - https://kandydacipb.edu.pl/rekr